# World Model Evaluator

> Fill in a module description here

In [ ]:
#| default_exp evaluators.world_model

In [ ]:
#| export
from collections import defaultdict
from typing import Any, Callable

import torch
import torch.nn.functional as F
import numpy as np
from einops import rearrange
import hydra
import wandb
from fastcore.utils import patch

from c3jepa_wm.utils import channel
from c3jepa_wm.utils.env_utils import MultiAgentEnvPool, set_env_state


In [ ]:
#| export
class MultiAgentGoalEvaluator:
    """
    Dataset-driven evaluation of the JEPA planner for a 2-agent communicative
    setting, driven through a `MultiAgentEnvPool` of N independent
    multi-agent envs -- one env instance per episode evaluated in parallel,
    mirroring `World._evaluate_from_dataset`'s `num_envs == len(episodes_idx)`
    pattern.

    For each agent, planning is now genuinely BATCHED across all N episodes
    in a chunk (a single `planner.plan(info[agent])` call handles all of
    them), rather than looping episodes one at a time at batch size 1.

    Args:
        model: JEPA model (has .encoder, .predictor, .action_encoder, .get_cost, .rollout)
        vqvae: sender-side VQ-VAE module; vqvae.encode(pixels) -> (B, 1, 49) token indices
        planner_cfg: hydra config to instantiate one planner per agent
        history_size: number of past steps fed as context (matches JEPA.rollout's history_size)
        goal_offset: number of steps ahead the goal is set
        agents: list of agent ids, must have length 2 (should match each
            env's `possible_agents`)
        device: torch device
        noop_action: action substituted for an agent that has reached its
            goal but whose env still expects an action from it every step.
    """

    def __init__(
        self,
        data_module,
        model,
        planner_cfg,
        slurm_jobid= None,
        history_size=3,
        goal_offset=10,
        dataset_agent_keys=None,
        agents=(0, 1),
        device="cpu",
        SNR=10.0,
        p_max=10,
        noise_power=1e-3,
        noop_action=3,
        budget=150,
    ):
        assert len(agents) == 2, "This evaluator only supports exactly 2 agents."
        self.data_module = data_module
        self.data_module.setup()

        self.model = model['jepa'].to(device).eval()
        self.vqvae = model['vqvae'].to(device).eval()

        # env-side ids (ints, e.g. 0/1 -- match PettingZooWrapper's agent.index)
        self.agents = list(agents)
        # dataset-side keys (strings, e.g. "agent_0"/"agent_1" -- match batch dict columns)
        self.dataset_agent_keys = (
            dict(dataset_agent_keys) if dataset_agent_keys is not None
            else {a: f"agent_{a}" for a in self.agents}
        )

        self.history_size = history_size
        self.goal_offset = goal_offset
        self.device = device
        self.SNR = SNR
        self.p_max = p_max
        self.noise_power = noise_power
        self.noop_action = noop_action
        self.budget = budget
        self.planners = {
            agent: hydra.utils.instantiate(planner_cfg, model=self.model, device=device)
            for agent in self.agents
        }


In [ ]:
#| export
@patch
@torch.no_grad()
def _encode_message(self: MultiAgentGoalEvaluator, partner_pixels_vqvae_t0, csi_t0, schedule=None, power=None, no_comm=False):
    """
    partner_pixels_vqvae_t0: (B, C, H, W) -- partner's obs at t0, VQ-VAE-transform space
    csi_t0: (B,) complex -- channel state at t0 for this sender->receiver link
    schedule, power: (B,) tensors, or None to skip channel entirely
    """
    partner_pixels_vqvae_t0 = partner_pixels_vqvae_t0.to(self.device)
    indices = self.vqvae.get_message_indices(partner_pixels_vqvae_t0)  # (B, 1, H, W)
    indices = rearrange(indices, "B H W -> B (H W)").long()  # (B, 49)

    if schedule is not None and power is not None:
        csi = csi_t0.to(self.device)
        schedule = torch.as_tensor(schedule, device=self.device)
        power = torch.as_tensor(power, device=self.device)

        indices = channel(
            schedule=schedule,
            power=power,
            msg_indices=indices,
            csi=csi,
            device=self.device,
            snr_db=self.SNR,
            no_comm=no_comm,
        )

    return indices.unsqueeze(1)  # (B, 1, 49)


In [ ]:
#| export
@patch
def _extract_power_and_schedule(self: MultiAgentGoalEvaluator, csi):
    snr_linear = 10 ** (self.SNR / 10.0)
    optimal_power = snr_linear * self.noise_power / (torch.abs(csi) ** 2 + 1e-8)
    schedule = (optimal_power <= self.p_max).float()
    return optimal_power, schedule


In [ ]:
#| export
@patch
def _build_agent_info_batch(self: MultiAgentGoalEvaluator, episodes: dict, agent, partner):
    """Build a batched info dict for `agent` across a chunk of episodes.

    episodes[agent][key]: already-windowed tensors with leading batch
    dim N (sliced to fixed size per episode at buffer-build time, then
    concatenated across the chunk) -- no per-episode t0 indexing needed
    here anymore.
    """
    pixels = episodes[agent]["pixels_hist"]
    actions = episodes[agent]["action_hist"]
    goal_pixels = episodes[agent]["goal_pixels"]

    partner_obs_vqvae_t0 = episodes[partner]["pov_vqvae_t0"]
    csi_t0 = episodes[partner]["csi_t0"].to(self.device)
    power_level, schedule = self._extract_power_and_schedule(csi_t0)
    msg_indices = self._encode_message(partner_obs_vqvae_t0, csi_t0, schedule=schedule, power=power_level)

    return {
        "pixels": pixels.to(self.device),
        "action": actions.to(self.device),
        "goal": goal_pixels.to(self.device),
        "msg_indices": msg_indices.to(self.device),
    }



In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_batch_fixed_t0(self: MultiAgentGoalEvaluator, episodes: dict, pool: MultiAgentEnvPool, t0s, max_steps=150):
    """
    Evaluate a chunk of N episodes IN PARALLEL, using receding-horizon MPC:
    at every real environment step, run one batched CEM plan per agent
    (covering all N episodes at once) and execute only the first action
    of each. Repeats for up to max_steps real steps.

    Args:
        episodes: dict with "goal_pos" (N, ...), "goal_obs" (N, num_agents, ...),
            and per-agent sub-dicts each holding (N, T, ...) tensors, as
            produced by stacking single-episode batches from the dataloader.
        pool: MultiAgentEnvPool with `pool.num_envs == N`.
        t0s: (N,) int array/tensor of start steps, one per episode.
        max_steps: real-env step budget.

    Messaging: partner message is used only for the very first planning
    call at t0; dropped for all subsequent replanning calls.
    """
    H = self.history_size
    N = pool.num_envs
    t0s = np.asarray(t0s)
    assert (t0s >= H - 1).all(), f"t0 too early for history_size={H}"

    # --- fixed goal embeddings, batched over N episodes, per agent ---
    goal_emb = {}
    for ai, agent in enumerate(self.agents):
        g = episodes["goal_obs"][:, ai].unsqueeze(1).to(self.device)  # (N, 1, C, H, W)
        enc = self.model.encode({"pixels": g})
        goal_emb[agent] = enc["emb"][:, 0]  # (N, D)

    # --- reset all N envs, then force each to its recorded state at its t0 ---
    pool.reset(seed=0)
    goal_pos_list = episodes["goal_pos"].tolist()
    for i in range(N):
        agent_positions = {
            a: episodes[a]["pos_t0"][i].tolist() for a in self.agents
        }
        agent_directions = {
            a: int(episodes[a]["dir_t0"][i].item()) for a in self.agents
        }
        set_env_state(pool, i, goal_pos_list[i], agent_positions, agent_directions, self.agents)

    # --- initial per-agent batched info (includes message, used once) ---
    info = {}
    for agent in self.agents:
        partner = [a for a in self.agents if a != agent][0]
        info[agent] = self._build_agent_info_batch(episodes, agent, partner)

    results = {agent: {"step": [], "embedding_dist_to_goal": [], "reached": []} for agent in self.agents}

    # per-agent, per-episode alive mask -- True while that agent in that
    # episode still needs stepping (mirrors World._run_iter's `alive`).
    alive = {agent: np.ones(N, dtype=bool) for agent in self.agents}

    for real_step in range(max_steps):
        # env-level mask: skip envs where BOTH agents are done, for efficiency
        env_mask = alive[self.agents[0]] | alive[self.agents[1]]
        if not env_mask.any():
            break

        actions = {}
        for agent in self.agents:
            # full-batch replan every step (planner itself is unchanged --
            # no per-episode slicing here, matching the current design);
            # done agents' actions get overwritten with noop below.
            _, plan = self.planners[agent].plan(info[agent])  # (N, horizon)
            act = plan[:, 0].cpu().numpy()
            act = np.where(alive[agent], act, self.noop_action)
            actions[agent] = act

        _, rewards, terminateds, truncateds, stacked_infos = pool.step(
            actions, mask=env_mask, noop_action=self.noop_action
        )

        for agent in self.agents:
            for i in range(N):
                if not alive[agent][i] or not env_mask[i]:
                    continue

                pov = stacked_infos[agent]["pov"][i, 0]
                frame = self.data_module.val_receiver_transform(pov).unsqueeze(0).unsqueeze(0).to(self.device)
                enc = self.model.encode({"pixels": frame})
                cur_emb = enc["emb"][:, 0]

                dist = F.mse_loss(cur_emb, goal_emb[agent][i:i + 1], reduction="none").sum(-1)
                reached = tuple(pool.envs[i].env.unwrapped.agents[agent].state.pos) == tuple(goal_pos_list[i])

                results[agent]["step"].append((i, real_step))
                results[agent]["embedding_dist_to_goal"].append(dist.item())
                results[agent]["reached"].append(reached)

                new_action = torch.tensor([[actions[agent][i]]], device=self.device)
                info[agent]["pixels"][i] = torch.cat(
                    [info[agent]["pixels"][i, 1:], frame[0]], dim=0
                )
                info[agent]["action"][i] = torch.cat(
                    [info[agent]["action"][i, 1:].unsqueeze(1), new_action], dim=0
                ).squeeze()

                if reached:
                    alive[agent][i] = False

        # msg_indices only used on the very first planning call
        for agent in self.agents:
            info[agent].pop("msg_indices", None)

    return results



In [ ]:
#| export
@patch
@torch.no_grad()
def evaluate_dataset_fixed_t0(self: MultiAgentGoalEvaluator, make_env: Callable[[], Any],
                               num_episodes=None, t0=None):
    """
    Gathers up to `num_episodes` valid episodes from the val dataset into
    ONE batch, builds a single `MultiAgentEnvPool` sized exactly to that
    batch, and evaluates all of them together in one call.

    This mirrors `World`: `World.__init__` builds `EnvPool` ONCE, sized to
    `num_envs`, and `_evaluate_from_dataset` requires
    `len(episodes_idx) == self.num_envs` -- it never internally re-chunks
    episodes into smaller groups. Memory-bounding for the expensive part
    (CEM candidate sampling + cost evaluation) is the PLANNER's
    responsibility now (`DiscreteCEMPlanner.batch_size`), exactly like
    `CEMSolver.batch_size` lives inside the solver, not inside `World`.
    Consequently there's no more `chunk_buffer` / `flush_chunk` / `_SubPool`
    here -- that machinery existed only to work around the planner lacking
    its own chunking, which is no longer the case.

    Args:
        make_env: zero-arg factory building one fresh multi-agent env
            instance. Called once per gathered episode.
        num_episodes: cap on how many valid episodes to gather. If None,
            gathers every valid episode in the val dataset (be mindful of
            memory: this builds one live env instance per episode).
        t0: if None, defaults to history_size - 1 per episode. If an int,
            used as a fixed absolute t0 for every episode (episodes too
            short for it are skipped).

    NOTE: assumes the dataloader yields batch_size=1 (single episode per
    batch) -- episodes are gathered here and stacked into one batch before
    being handed to the pool/planner.
    """
    dataset = self.data_module.val_dataloader()
    assert dataset.batch_size == 1, "evaluate_dataset_fixed_t0 requires the dataloader's own batch_size=1"

    H = self.history_size
    max_steps = self.budget
    episodes_list = []

    for i, batch in enumerate(dataset):
        if num_episodes is not None and len(episodes_list) >= num_episodes:
            break

        length = batch["length"][0].item()
        this_t0 = (H - 1) if t0 is None else t0
        if this_t0 >= length:
            print(f"Skipping episode {i}: length={length} too short for t0={this_t0}")
            continue

        episode = {
            "goal_pos": batch["goal_pos"][0],
            "goal_obs": batch["goal_obs"][0],
            "t0": this_t0,
        }
        for agent in self.agents:
            a_batch = batch[self.dataset_agent_keys[agent]]
            episode[agent] = {
                "pixels_hist": a_batch["pixels"][0:1, this_t0 - H + 1: this_t0 + 1],
                "action_hist": a_batch["action"][0:1, this_t0 - H + 1: this_t0 + 1],
                # offset-based hindsight goal -- this is what the planner is
                # conditioned on, matching World's _extract_init_goal pattern
                "goal_pixels": a_batch["pixels"][0:1, this_t0 + self.goal_offset],
                "pos_t0": a_batch["pos"][0:1, this_t0],
                "dir_t0": a_batch["dir"][0:1, this_t0],
                "pov_vqvae_t0": a_batch["pov_seq_vqvae"][0:1, this_t0],
                "csi_t0": a_batch["csi"][0:1, this_t0],
            }
        episodes_list.append(episode)

    n = len(episodes_list)
    if n == 0:
        raise ValueError("No valid episodes found for evaluation.")

    # -- assemble the single, full-size batch (mirrors _extract_init_goal
    # gathering all N episodes' init/goal state before World ever resets
    # a single env) --
    episodes = {
        "goal_pos": torch.stack([e["goal_pos"] for e in episodes_list]),
        "goal_obs": torch.stack([e["goal_obs"] for e in episodes_list]),
    }
    for agent in self.agents:
        episodes[agent] = {
            k: torch.cat([e[agent][k] for e in episodes_list], dim=0)
            for k in episodes_list[0][agent]
        }
    t0s = np.array([e["t0"] for e in episodes_list])

    # -- ONE pool, sized exactly to n, built once -- mirrors World.__init__'s
    # fixed num_envs. No per-chunk pool rebuilding. --
    pool = MultiAgentEnvPool([make_env for _ in range(n)], agents=self.agents)

    per_agent_step_dist = {agent: defaultdict(list) for agent in self.agents}
    per_agent_step_success = {agent: defaultdict(list) for agent in self.agents}

    print(f"Evaluating {n} episodes in parallel (max_steps={max_steps})...")
    ep_res = self.evaluate_batch_fixed_t0(episodes, pool, t0s, max_steps=max_steps)

    for agent in self.agents:
        for (i, step), dist, reached in zip(
            ep_res[agent]["step"], ep_res[agent]["embedding_dist_to_goal"], ep_res[agent]["reached"]
        ):
            per_agent_step_dist[agent][step].append(dist)
            per_agent_step_success[agent][step].append(float(reached))

    curves = {}
    for agent in self.agents:
        steps = sorted(per_agent_step_dist[agent].keys())
        curves[agent] = {
            "step": steps,
            "mean_embedding_dist": [float(torch.tensor(per_agent_step_dist[agent][s]).mean()) for s in steps],
            "std_embedding_dist": [float(torch.tensor(per_agent_step_dist[agent][s]).std()) for s in steps],
            "success_rate": [float(torch.tensor(per_agent_step_success[agent][s]).mean()) for s in steps],
        }

    return curves
